In [6]:
# Après avoir créé le client
print("Collections disponibles :", client.list_collections())

# On peut aussi essayer de récupérer la collection directement
try:
    collection = client.get_collection(model_config["vector_db"]["collection_name"])
    print(f"Collection trouvée : {collection.name}, nombre de documents : {collection.count()}")
except Exception as e:
    print(f"Erreur : {e}")

Collections disponibles : [Collection(name=cuad_contracts)]
Collection trouvée : cuad_contracts, nombre de documents : 0


In [7]:
import sys
import yaml
from pathlib import Path

sys.path.append(".")

import chromadb
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# On met le chemin ABSOLU en dur pour être sûr à 100%
chroma_folder = Path(r"C:\Users\Errachdi\OneDrive - Institut National de Statistique et d'Economie Appliquee\Bureau\projet saad\legal-rag-assistant\outputs\chroma_db")

print(f"📂 Dossier de la base : {chroma_folder}")

# Création du client avec le chemin absolu
client = chromadb.PersistentClient(path=str(chroma_folder))

# 1. On regarde toutes les collections existantes
print("\n📋 Liste de TOUTES les collections dans cette base :")
collections = client.list_collections()
for col in collections:
    print(f"   - {col.name} (documents: {col.count()})")

# 2. On récupère la collection spécifique
collection_name = "cuad_contracts"  # On met le nom en dur
print(f"\n🔑 On essaie de récupérer la collection : '{collection_name}'")

try:
    collection = client.get_collection(collection_name)
    print(f"✅ COLLECTION TROUVÉE !")
    print(f"   Nombre total de chunks : {collection.count()}")
    
    # On charge le modèle d'embedding pour LangChain
    with open("../configs/model_config.yaml", "r") as f:
        model_config = yaml.safe_load(f)
    
    embedding_model = HuggingFaceEmbeddings(
        model_name=model_config["embedding"]["model_name"],
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
    
    # On crée le vector_store avec cette collection
    vector_store = Chroma(
        client=client,
        collection_name=collection_name,
        embedding_function=embedding_model
    )
    
    print("\n🚀 Test de recherche (Governing Law)...")
    results = vector_store.similarity_search("Governing Law", k=3)
    
    if results:
        print(f"✅ {len(results)} résultats trouvés !")
        for i, doc in enumerate(results):
            print(f"\n📄 Résultat {i+1}")
            print(f"  Titre : {doc.metadata.get('header', 'Inconnu')}")
            print(f"  Extrait : {doc.page_content[:200]}...")
    else:
        print("❌ Aucun résultat.")

except Exception as e:
    print(f"❌ Erreur : {e}")
    print("\n💡 Suggestion : Si vous voyez 'Collection not found', essayez de remplacer 'cuad_contracts' par le nom affiché dans la liste ci-dessus.")

📂 Dossier de la base : C:\Users\Errachdi\OneDrive - Institut National de Statistique et d'Economie Appliquee\Bureau\projet saad\legal-rag-assistant\outputs\chroma_db

📋 Liste de TOUTES les collections dans cette base :
   - cuad_contracts (documents: 25772)

🔑 On essaie de récupérer la collection : 'cuad_contracts'
✅ COLLECTION TROUVÉE !
   Nombre total de chunks : 25772


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


🚀 Test de recherche (Governing Law)...
✅ 3 résultats trouvés !

📄 Résultat 1
  Titre : Section 19.1
  Extrait : (Governing Law), and...

📄 Résultat 2
  Titre : Section 10.2
  Extrait : (Governing Law) through...

📄 Résultat 3
  Titre : Section 8.01
  Extrait : Governing Law 19...


In [8]:
def test_query(query_text, k=5):
    print(f"\n--- REQUÊTE : {query_text} ---\n")
    
    results_with_scores = vector_store.similarity_search_with_score(query_text, k=k)
    
    for i, (doc, score) in enumerate(results_with_scores):
        print(f"Résultat {i+1} (Score de distance : {score:.4f})")
        print(f"  Fichier : {doc.metadata.get('source', 'Inconnu')}")
        print(f"  Titre   : {doc.metadata.get('header', 'Inconnu')}")
        print(f"  Extrait : {doc.page_content[:250].replace(chr(10), ' ')}...")
        print("-" * 80)

In [9]:
test_query("Quelle est la loi qui régit ce contrat ?", k=5)
test_query("Durée du contrat et date d'expiration", k=5)
test_query("Clause de confidentialité et protection des données", k=5)
test_query("Comment se fait le paiement ?", k=5)


--- REQUÊTE : Quelle est la loi qui régit ce contrat ? ---

Résultat 1 (Score de distance : 1.3305)
  Fichier : ElPolloLocoHoldingsInc_20200306_10-K_EX-10.16_12041700_EX-10.16_Development Agreement.txt
  Titre   : 16.2.
  Extrait : At the election of the Franchisor, the provisions of this Section 16 shall not apply to controversies relating to any fee due the Franchisor by Developer or its affiliates, any promissory note payments due the Franchisor by Developer, or any trade pa...
--------------------------------------------------------------------------------
Résultat 2 (Score de distance : 1.3602)
  Fichier : OFGBANCORP_03_28_2007-EX-10.23-OUTSOURCING AGREEMENT.txt
  Titre   : 17.11.
  Extrait : Force Majeure. Notwithstanding any provision contained in this Agreement, neither party shall be liable to the other to the extent fulfillment or performance if any terms or provisions of this Agreement is delayed or prevented by revolution or other ...
--------------------------------------